# Colab Phase 2 — ModernBERT-Ja 70m 本番学習

`docs/guides/COLAB_TRAIN_GUIDE.md` + `docs/reranker/tasks/NEXT_TASK_PHASE2.md` 準拠。

**鉄則**: torch は pip しない / 依存追加後は一度再起動 / Drive に保存。

前提: Drive に `Mozc-Ai-Training`（少なくとも `tools/` と `data/rerank_v3/`）があること。

In [ ]:
# Cell 0 — GPU / 既設 torch 確認（何も入れない）
!nvidia-smi -L
import torch, sys
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda_available", torch.cuda.is_available())
print("cuda", torch.version.cuda)
assert torch.cuda.is_available(), "ランタイムを GPU に変更してください"

In [ ]:
# Cell 1 — transformers 系だけ上乗せ（torch は触らない）
!pip install -q "transformers>=4.48,<5" "tokenizers>=0.21" sentencepiece accelerate
import transformers, importlib.metadata as m
print("transformers", transformers.__version__, "/ hub", m.version("huggingface_hub"))

### Cell 2 — ランタイム再起動（依存を入れた直後・一度だけ）
次のセルを実行 → 再接続後は **Cell 0 だけ再実行** → Cell 1/2 は飛ばして Cell 3 へ。

In [ ]:
# Cell 2 — 再起動（初回のみ）。再接続後は実行しない。
import os
os.kill(os.getpid(), 9)

In [ ]:
# Cell 3 — Drive マウント + リポルートへ
from google.colab import drive
drive.mount('/content/drive')

import os
# HF キャッシュを Drive へ（再ダウンロード回避）
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ★ Drive 上の実際のパスに合わせて変更
REPO = '/content/drive/MyDrive/Mozc-Ai-Training'
%cd $REPO
!ls tools/rerank data/rerank_v3
!wc -l data/rerank_v3/train.jsonl data/rerank_v3/holdout.jsonl

In [ ]:
# Cell 4 — スモーク（モデルがロードできるか）
import torch
from transformers import AutoModel, AutoTokenizer

NAME = 'sbintuitions/modernbert-ja-70m'  # Phase1 継続。ruri に替えるならここだけ
tok = AutoTokenizer.from_pretrained(NAME, trust_remote_code=True)
try:
    mdl = AutoModel.from_pretrained(NAME, trust_remote_code=True, torch_dtype=torch.float32).cuda().eval()
except Exception as e:
    print('retry with sdpa:', type(e).__name__, e)
    mdl = AutoModel.from_pretrained(
        NAME, trust_remote_code=True, torch_dtype=torch.float32, attn_implementation='sdpa'
    ).cuda().eval()
enc = tok('読み: とうきょうと [SEP] 候補: 東京都', return_tensors='pt', truncation=True, max_length=48).to('cuda')
with torch.no_grad():
    out = mdl(**enc)
print('OK hidden:', tuple(out.last_hidden_state.shape))
del mdl
torch.cuda.empty_cache()

In [ ]:
# Cell 5 — Phase2 学習（T4: batch 64 から。OOM なら 32）
# A100 なら 128〜256 も可。ローカル ROCm 実績は batch 1280 級だが Colab T4 では不可。
!python -m tools.rerank.train_cross_encoder train \
  --train data/rerank_v3/train.jsonl \
  --eval  data/rerank_v3/holdout.jsonl \
  --model sbintuitions/modernbert-ja-70m \
  --out   artifacts/rerank/modernbert70m_ce_v3 \
  --epochs 2 --batch-size 64 --max-len 128 --max-neg 15 \
  --fp16 --grad-checkpointing --require-cuda --require-gold-in-nbest

In [ ]:
# Cell 6 — 成果確認（Drive 上ならこのまま残る）
!ls -lah artifacts/rerank/modernbert70m_ce_v3 | head
!du -sh artifacts/rerank/modernbert70m_ce_v3

In [ ]:
# Cell 7 — 動いた構成を lock（出力を docs/colab_requirements.lock.txt に貼る）
!pip freeze | grep -Ei '^(torch|transformers|tokenizers|sentencepiece|accelerate|numpy)=='